In [1]:
import os
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm
import torch
from transformer_lens import HookedTransformer, patching
import plotly.express as px
import gc
import datetime
import contextlib
import math

from collections import defaultdict
from torch.utils.data import DataLoader, Dataset
from eap.attribute import attribute
from eap.evaluate import evaluate_graph
from eap.graph import Graph

def _find_animacy_root(start: Path) -> Path:
    resolved = start.resolve()
    for base in (resolved, *resolved.parents):
        for candidate in (base, base / "animacy-circuit"):
            if (
                (candidate / "dataset").is_dir()
                and (candidate / "results").is_dir()
                and (candidate / "scripts").is_dir()
            ):
                return candidate
    raise FileNotFoundError("Could not locate the animacy-circuit project root.")

project_root = _find_animacy_root(Path.cwd())
executable_dir = project_root / "scripts" / "executable"
if str(executable_dir) not in sys.path:
    sys.path.insert(0, str(executable_dir))

notebook_dir = project_root / "scripts" / "notebooks"
working_dir = str(project_root)

In [3]:
dataset = pd.read_csv(f"{working_dir}/dataset/semantic_meaningful/metric_filtered/mp_filtered_avg_LD_pairs.csv")[["clean", "corrupt"]]

dataset.columns = ["clean_prefix", "corrupt_prefix"]


with open(Path(working_dir) / "dataset" / "semantic_meaningful" / "wordnet_lexname_targets_500x500.json") as f:
    data = json.load(f)

targets_raw = pd.json_normalize(data["targets"])

blacklist = json.loads((Path(working_dir) / "dataset" / "semantic_meaningful" / "blacklist.json").read_text(encoding="utf-8"))
blocked = {
      "animate": set(blacklist["animate"]["remove_now"]) | set(blacklist["animate"]["strongly_consider_removing"]),
      "inanimate": set(blacklist["inanimate"]["remove_now"]) | set(blacklist["inanimate"]["strongly_consider_removing"]),
  }

animate_words_list = [
    w for w in targets_raw.loc[0, "animate"]
    if w not in blocked["animate"]
]

inanimate_words_list = [
    w for w in targets_raw.loc[0, "inanimate"]
    if w not in blocked["inanimate"]
]

targets = pd.DataFrame({
      "animate": animate_words_list,
      "inanimate": inanimate_words_list,
  })

print(len(animate_words_list), len(inanimate_words_list))

device = "cuda" if torch.cuda.is_available() else "cpu"

model = HookedTransformer.from_pretrained("gpt2", device=device)
model.cfg.default_prepend_bos = True
tokenizer = model.tokenizer
DEVICE = model.cfg.device
BATCH_SIZE = 50
resid_stream_path_file = os.path.join(working_dir, "results", "2026-05-08", "Residual_Stream_Patching_2026-05-08.pt")

mlp_results_file_verb = os.path.join(working_dir, "results", "2026-05-08", "MLP_Outputs_Targeted_Verb_2026-05-08.pt")
mlp_results_file_by = os.path.join(working_dir, "results", "2026-05-08", "MLP_Outputs_Targeted_by_2026-05-08.pt")
mlp_results_file_the = os.path.join(working_dir, "results", "2026-05-08", "MLP_Outputs_Targeted_the_2026-05-08.pt")

attn_results_file_verb = os.path.join(working_dir, "results", "2026-05-08", "Attention_Outputs_Targeted_Verb_2026-05-08.pt")
attn_results_file_by = os.path.join(working_dir, "results", "2026-05-08", "Attention_Outputs_Targeted_by_2026-05-08.pt")
attn_results_file_the = os.path.join(working_dir, "results", "2026-05-08", "Attention_Outputs_Targeted_the_2026-05-08.pt")


450 450


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2 into HookedTransformer


In [4]:
print(model)

HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
      (h

In [5]:
dataset

,clean_prefix,corrupt_prefix
0,The tram was parked by the,The tram was crushed by the
1,The terminal was renovated by the,The terminal was damaged by the
2,The citizen was transported by the,The citizen was struck by the
3,The soldier was examined by the,The soldier was trapped by the
4,The victim was examined by the,The victim was blinded by the
...,...,...
5440,The canoe was painted by the,The canoe was blocked by the
5441,The survivor was protected by the,The survivor was frozen by the
5442,The car was inspected by the,The car was flooded by the
5443,The hangar was repaired by the,The hangar was wrecked by the


In [6]:
targets

,animate,inanimate
0,officer,dust
1,manager,tsunami
2,captain,earthquake
3,author,sunlight
4,farmer,moisture
...,...,...
445,foster,shortcut
446,fuller,shrine
447,fundamentalist,silicone
448,geek,snapshot


In [7]:
def create_target_tensor(word_list, tokenizer, device):
    valid_ids = []
    
    for word in word_list:
        # 1. Prepend a space to match how it appears in the sentence
        # 2. add_special_tokens=False prevents the tokenizer from adding <bos>
        tokens = tokenizer(" " + word, add_special_tokens=False).input_ids
        
        # 3. Safety Check: Ensure the word hasn't been split into multiple subwords
        if len(tokens) != 1:
            print(f"WARNING: The word '{word}' split into {len(tokens)} tokens {tokens}. Skipping.")
            continue
            
        valid_ids.append(tokens[0])
        
    # 4. Convert to a PyTorch tensor and move it to the GPU
    return torch.tensor(valid_ids, dtype=torch.long, device=device)

# Generate the tensors
animate_ids_tensor = create_target_tensor(animate_words_list, tokenizer, DEVICE)
inanimate_ids_tensor = create_target_tensor(inanimate_words_list, tokenizer, DEVICE)

print(animate_ids_tensor.shape, inanimate_ids_tensor.shape)

torch.Size([450]) torch.Size([450])


In [8]:
def verify_target_tensors(original_list, ids_tensor, tokenizer):
    """
    One-time safety check: Decodes the tensor back to strings and ensures 
    it strictly matches the valid words from the original list.
    """
    decoded_words = [tokenizer.decode([token_id]) for token_id in ids_tensor]
    
    # We strip the prepended space from the decoded words for comparison
    decoded_clean = [word.strip() for word in decoded_words]
    original_clean = [word.strip() for word in original_list]
    
    # We expect the lengths to differ ONLY by the subwords we explicitly skipped earlier
    # But every word in the decoded list MUST exist in the original list.
    for word in decoded_clean:
        assert word in original_clean, f"FATAL ALIGNMENT ERROR: Tokenizer mapped an ID to '{word}', which is not in our target set!"
        
    print(f"Vocab Alignment Verified: Tensor strictly maps to {len(decoded_clean)} target concepts.")

def get_input_ids_with_bos(text_or_texts, model):
    tokens = model.to_tokens(
        text_or_texts,
        prepend_bos=True,
        move_to_device=False,
    )
    return tokens[0] if isinstance(text_or_texts, str) else tokens

# Run this ONCE in your notebook after creating the tensors
verify_target_tensors(animate_words_list, animate_ids_tensor, tokenizer)
verify_target_tensors(inanimate_words_list, inanimate_ids_tensor, tokenizer)

Vocab Alignment Verified: Tensor strictly maps to 450 target concepts.
Vocab Alignment Verified: Tensor strictly maps to 450 target concepts.


In [9]:
valid_indices = []

for idx, row in dataset.iterrows():
    clean_prefix = row["clean_prefix"]
    corrupt_prefix = row["corrupt_prefix"]
    
    clean_prefix_toks = get_input_ids_with_bos(clean_prefix, model)
    corrupt_prefix_toks = get_input_ids_with_bos(corrupt_prefix, model)
    
    # The prefixes must be the exact same token length to align the patching positions
    if len(clean_prefix_toks) == len(corrupt_prefix_toks):
        valid_indices.append(idx)

# Create the aligned dataset
aligned_dataset = dataset.loc[valid_indices].reset_index(drop=True)

# PRECOMPUTE seq_len HERE so compute_sselection_accuracy can use it
aligned_dataset['seq_len'] = aligned_dataset['clean_prefix'].apply(lambda x: len(get_input_ids_with_bos(x, model)))

print(f"Filtered dataset size: {len(aligned_dataset)}")
print(f"Removed {len(dataset) - len(aligned_dataset)} misaligned pairs.")

Filtered dataset size: 5445
Removed 0 misaligned pairs.


<h3>Deprecated</h3>
Since I have sentences of different length in my dataset I need to group them in batches of sentences of the same length, we cannot use LEFT because we want to avoid attention poisoning

In [10]:
def generate_exact_length_batches(df, tokenizer, batch_size=16, device="cpu"):
    # Simply group by the precomputed column! No .copy() or .apply() needed here.
    grouped = df.groupby('seq_len')
    
    for length, group in grouped:
        for i in range(0, len(group), batch_size):
            batch_df = group.iloc[i : i + batch_size]
            
            clean_tokens = get_input_ids_with_bos(
                batch_df["clean_prefix"].tolist(),
                model,
            ).to(device)
            
            corrupt_tokens = get_input_ids_with_bos(
                batch_df["corrupt_prefix"].tolist(),
                model,
            ).to(device)
            
            # --- SAFETY CHECKS ---
            # 1. Ensure clean and corrupt batches are exactly the same shape
            assert clean_tokens.shape == corrupt_tokens.shape, \
                f"Shape mismatch! Clean: {clean_tokens.shape}, Corrupt: {corrupt_tokens.shape}"
            
            # 2. Ensure the actual tensor length matches the pandas grouped length
            assert clean_tokens.shape[1] == length, \
                f"Length mismatch! Expected {length}, got {clean_tokens.shape[1]}"
            
            yield clean_tokens, corrupt_tokens, batch_df

In [11]:
def get_per_sentence_ppl(logits, input_ids, attention_mask):
    # Remove the last token predicted from the logits which is the <eos> token
    # Remove also the first token from the input_ids which is never predicted, to align them for loss calculation
    # Shift the attention mask as well to align with the labels
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = input_ids[..., 1:].contiguous()
    shift_mask = attention_mask[..., 1:].contiguous()  

    # Calculate Cross Entropy for every single token
    loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
    #The loss function wants [tot_tokens, vocab_size] and [tot_tokens], so we flatten the first two dimensions of both tensors
    #shift.logits.size(-1) is the vocab size, so we keep that dimension intact and flatten the rest
    loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

    #then we reshape the loss back to [batch_size, seq_len] to get the average loss per sentence
    # apply the attention mask to ignore padding tokens in the loss calculation
    loss = loss.view(shift_labels.size()) * shift_mask  

    # Now we sum the loss for each sentence and divide by the number of tokens (sum of attention mask) to get the average loss per sentence
    avg_loss_per_sentence = loss.sum(dim=1) / shift_mask.sum(dim=1)

    return torch.exp(avg_loss_per_sentence)

def average_logit_difference_from_final_logits(final_logits, animate_ids_tensor, inanimate_ids_tensor):
    anim_logits = final_logits[:, animate_ids_tensor]
    inan_logits = final_logits[:, inanimate_ids_tensor]
    return anim_logits.mean(dim=-1) - inan_logits.mean(dim=-1)

def final_token_average_logit_difference(logits: torch.Tensor, animate_ids_tensor: torch.Tensor, inanimate_ids_tensor: torch.Tensor) -> torch.Tensor:
    assert logits.ndim == 3, f"Expected 3D logits, got {logits.ndim}D tensor of shape {logits.shape}"
    logit_diff = average_logit_difference_from_final_logits(logits[:, -1, :], animate_ids_tensor, inanimate_ids_tensor)
    assert not torch.isnan(logit_diff).any(), "NaNs generated during logit difference calculation!"
    return logit_diff

def compute_animacy_accuracy(df, model, tokenizer, animate_ids_tensor, inanimate_ids_tensor, batch_size=16):
    # Dictionary to store {original_dataframe_index: boolean_accuracy_result}
    results_dict = {}
    
    batch_generator = generate_exact_length_batches(df, tokenizer, batch_size=batch_size, device=model.cfg.device)
    estimated_batches = sum(math.ceil(len(g) / batch_size) for _, g in df.groupby('seq_len'))
    
    for clean_tokens, corrupt_tokens, batch_df in tqdm(batch_generator, total=estimated_batches, desc="Evaluating Batches"):
        
        with torch.no_grad():
            logits_good = model(clean_tokens)
            logits_bad = model(corrupt_tokens)
            
        # Extract the logits for the VERY LAST token of the prefix
        logits_good = logits_good[:, -1, :]
        logits_bad = logits_bad[:, -1, :]
        
        good_metric = average_logit_difference_from_final_logits(logits_good, animate_ids_tensor, inanimate_ids_tensor)
        bad_metric = average_logit_difference_from_final_logits(logits_bad, animate_ids_tensor, inanimate_ids_tensor)
        
        batch_acc = ((good_metric > 0) & (bad_metric < 0)).cpu().tolist()
        
        # Map results back to their original dataframe indices
        for original_idx, acc in zip(batch_df.index, batch_acc):
            results_dict[original_idx] = acc
            
    # Reconstruct the mask list in the exact order of the original dataframe
    ordered_mask = [results_dict[idx] for idx in df.index]
        
    return ordered_mask, np.mean(ordered_mask)

In [12]:
mask, accuracy = compute_animacy_accuracy(
    aligned_dataset, 
    model, 
    tokenizer, 
    animate_ids_tensor, 
    inanimate_ids_tensor, 
    batch_size=BATCH_SIZE
)

print(f"Dataset Baseline Accuracy: {accuracy:.2%}")

# Filter the dataframe to ONLY keep the pairs where the model succeeded
filtered_patching_dataset = aligned_dataset[mask].reset_index(drop=True)
print(f"Kept {len(filtered_patching_dataset)} perfect pairs for patching.")

Evaluating Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Dataset Baseline Accuracy: 100.00%
Kept 5445 perfect pairs for patching.


In [13]:
# Filter the dataset using the mask
dataset_filtered = aligned_dataset[mask].copy().reset_index(drop=True).sample(n=500, random_state=42)  # Shuffle the filtered dataset

print(f"Original dataset size: {len(aligned_dataset)}")
print(f"Filtered dataset size: {len(dataset_filtered)}")


Original dataset size: 5445
Filtered dataset size: 500


In [12]:
# Filter the dataset using the mask
dataset_filtered = aligned_dataset.copy().reset_index(drop=True).sample(n=500, random_state=42)  # Shuffle the filtered dataset

print(f"Original dataset size: {len(aligned_dataset)}")
print(f"Filtered dataset size: {len(dataset_filtered)}")

Original dataset size: 5445
Filtered dataset size: 500


Might the filtering above introduce survivorship bias?

After having filtered the dataset let's start by doing activation aptching on the residual stream, the idea is to narrow down the essential unit by first identifying the layer important for the task, then the specific attention/mlp heads, then specific edges and finally the paths

First we need a metric to quantify the model direct relative preference for the correct concept over the incorrect concept. This metric is the logit difference

In [13]:
def make_avg_logit_difference_recovery_metric(animate_ids_tensor: torch.Tensor, inanimate_ids_tensor: torch.Tensor, eps: float = 1e-6):
    def metric_factory(clean_logits: torch.Tensor, corrupt_logits: torch.Tensor):
        clean_logit_diff = final_token_average_logit_difference(clean_logits, animate_ids_tensor, inanimate_ids_tensor)
        corrupt_logit_diff = final_token_average_logit_difference(corrupt_logits, animate_ids_tensor, inanimate_ids_tensor)

        denominator = clean_logit_diff - corrupt_logit_diff
        
        if not torch.all(denominator > eps):
            raise ValueError(
                "Each sample must have a sufficiently positive clean-corrupt "
                "logit-difference margin for normalized patching."
            )

        def metric(logits: torch.Tensor) -> torch.Tensor:
            patched_logit_diff = final_token_average_logit_difference(logits, animate_ids_tensor, inanimate_ids_tensor)
            normalized_recovery = (patched_logit_diff - corrupt_logit_diff) / denominator

            if not torch.isfinite(normalized_recovery).all():
                raise ValueError("Non-finite values generated during normalized recovery calculation.")

            return normalized_recovery.mean()

        return metric, clean_logit_diff.mean().item(), corrupt_logit_diff.mean().item()

    return metric_factory

<h3>Activation patching on the residual stream</h3>

To do so we need to run the correct sentence and save each activation on the cache

In [14]:
def _standardize_patch_result_shape(batch_patch_results: torch.Tensor, seq_len: int, n_heads: int, ) -> torch.Tensor:
    """
    Accepted shapes:
    - [layers, pos]
    - [layers, pos, heads]  -> converted to [layers, heads, pos]
    - [layers, heads, pos]  -> left unchanged
    """
    if batch_patch_results.ndim == 2:
        if batch_patch_results.shape[1] != seq_len:
            raise ValueError(
                f"2D patch result must have shape [layers, pos] with pos={seq_len}, "
                f"got {tuple(batch_patch_results.shape)}"
            )
        return batch_patch_results

    if batch_patch_results.ndim == 3:
        if batch_patch_results.shape[1] == seq_len and batch_patch_results.shape[2] == n_heads:
            return batch_patch_results.permute(0, 2, 1)

        if batch_patch_results.shape[1] == n_heads and batch_patch_results.shape[2] == seq_len:
            return batch_patch_results

        raise ValueError(
            "3D patch result must have shape [layers, pos, heads] or [layers, heads, pos]. "
            f"Expected seq_len={seq_len}, n_heads={n_heads}, got {tuple(batch_patch_results.shape)}"
        )

    raise ValueError(
        f"Patch result must be 2D or 3D, got ndim={batch_patch_results.ndim} "
        f"with shape {tuple(batch_patch_results.shape)}"
    )


def batched_exact_patching(model, df, tokenizer, patching_func, patching_metric_factory, names_filter, batch_size: int = 16, requires_grad: bool = False, safety_checks: bool = False):
    total_samples = len(df)
    max_seq_len = df["seq_len"].max()

    total_clean_metric = 0.0
    total_corrupt_metric = 0.0

    total_patch_heatmap = None
    total_counts = None

    print(f"Starting batched patching over {total_samples} samples. Max sequence length: {max_seq_len}")

    context = torch.no_grad() if not requires_grad else contextlib.nullcontext()
    with context:
        func_name = getattr(patching_func, "__name__", getattr(getattr(patching_func, "func", None), "__name__", "Function"))
        grouped = df.groupby("seq_len")

        with tqdm(total=total_samples, desc=f"Patching {func_name}", unit="seq") as pbar:
            for length, group in grouped:
                if safety_checks:
                    sample_text = group.iloc[0]["clean_prefix"]
                    sample_tokens = get_input_ids_with_bos(
                        sample_text,
                        model,
                    )
                    final_input_token_str = tokenizer.decode(sample_tokens[-1])
                    print(f"[DEBUG] Group Length {length} | Example: '{sample_text}'")
                    print(
                        "[DEBUG]        ---> logits[:, -1, :] predicts the NEXT token after "
                        f"the final input token '{final_input_token_str}'\n"
                    )

                for i in range(0, len(group), batch_size):
                    batch_df = group.iloc[i : i + batch_size]
                    actual_batch_size = len(batch_df)

                    clean_batch = get_input_ids_with_bos(
                        batch_df["clean_prefix"].tolist(),
                        model,
                    ).to(model.cfg.device)

                    corrupt_batch = get_input_ids_with_bos(
                        batch_df["corrupt_prefix"].tolist(),
                        model,
                    ).to(model.cfg.device)

                    if clean_batch.shape != corrupt_batch.shape:
                        raise ValueError(
                            f"Shape mismatch between clean and corrupt batches: "
                            f"{tuple(clean_batch.shape)} vs {tuple(corrupt_batch.shape)}"
                        )

                    if clean_batch.shape[1] != length:
                        raise ValueError(
                            f"Expected tokenized batch length {length}, got {clean_batch.shape[1]}"
                        )

                    clean_logits, clean_cache = model.run_with_cache(clean_batch, names_filter=names_filter)
                    corrupt_logits = model(corrupt_batch)

                    batch_metric, batch_clean_base, batch_corrupt_base = patching_metric_factory(clean_logits, corrupt_logits)

                    total_clean_metric += batch_clean_base * actual_batch_size
                    total_corrupt_metric += batch_corrupt_base * actual_batch_size

                    batch_patch_results = patching_func(
                        model=model,
                        corrupted_tokens=corrupt_batch,
                        clean_cache=clean_cache,
                        patching_metric=batch_metric,
                    )

                    batch_patch_results = _standardize_patch_result_shape(
                        batch_patch_results=batch_patch_results,
                        seq_len=length,
                        n_heads=model.cfg.n_heads,
                    )

                    if total_patch_heatmap is None:
                        target_shape = list(batch_patch_results.shape)
                        target_shape[-1] = max_seq_len
                        total_patch_heatmap = torch.zeros(target_shape, device=batch_patch_results.device, dtype=batch_patch_results.dtype)
                        total_counts = torch.zeros(target_shape, device=batch_patch_results.device, dtype=batch_patch_results.dtype)

                    total_patch_heatmap[..., max_seq_len - length :] += (batch_patch_results * actual_batch_size)
                    total_counts[..., max_seq_len - length :] += actual_batch_size

                    del clean_logits, corrupt_logits, clean_cache, batch_patch_results
                    gc.collect()
                    torch.cuda.empty_cache()

                    pbar.update(actual_batch_size)

    final_clean_metric = total_clean_metric / total_samples
    final_corrupt_metric = total_corrupt_metric / total_samples

    counts_safe = torch.where(total_counts == 0, torch.ones_like(total_counts), total_counts)
    final_normalized_heatmap = total_patch_heatmap / counts_safe

    if not torch.isfinite(final_normalized_heatmap).all():
        print("CRITICAL WARNING: Non-finite values detected in the final heatmap!")

    if final_clean_metric <= final_corrupt_metric:
        print(
            f"WARNING: Clean baseline ({final_clean_metric:.2f}) is worse than or equal to "
            f"corrupt baseline ({final_corrupt_metric:.2f}). Check your metric."
        )

    return final_normalized_heatmap, final_clean_metric, final_corrupt_metric

In [15]:
def run_and_save_experiment(experiment_name, df, tokenizer, patching_func, metric_factory, filter_str, batch_size=16, safety_checks=False):
    print(f"--- Starting Experiment: {experiment_name} ---")
    
    normalized_map, clean_metric_mean, corrupt_metric_mean = batched_exact_patching(
        model=model,
        df=df,
        tokenizer=tokenizer,
        patching_func=patching_func,
        patching_metric_factory=metric_factory,
        names_filter=lambda name: name.endswith(filter_str),
        batch_size=batch_size,
        safety_checks=safety_checks
    )
    
    # Grab the tokens of the LONGEST sequence to act as the x-axis labels
    max_idx = df['seq_len'].idxmax()
    sample_tokens = get_input_ids_with_bos(
        df.loc[max_idx, "clean_prefix"],
        model,
    )
    
    experiment_data = {
        "experiment": experiment_name,
        "timestamp": datetime.datetime.now().strftime("%Y-%m-%d"),
        "normalized_heatmap": normalized_map.cpu(),
        "clean_metric_mean": clean_metric_mean,
        "corrupt_metric_mean": corrupt_metric_mean,
        "labels": [model.to_string(t) for t in sample_tokens]
    }
    
    save_dir = Path(working_dir) / "results" / experiment_data["timestamp"]
    save_dir.mkdir(parents=True, exist_ok=True)
    save_path = save_dir / f"{experiment_name}_{experiment_data['timestamp']}.pt"
    torch.save(experiment_data, save_path)
    print(f"Saved successfully to {save_path}\n")

    return save_path

In [16]:
def get_saved_normalized_heatmap(data):
    if "normalized_heatmap" in data:
        return data["normalized_heatmap"]
    return (data["raw_heatmap"] - data["corrupt_baseline"]) / (data["clean_baseline"] - data["corrupt_baseline"])

def get_saved_heatmap_x_labels(data):
    raw_labels = data.get("labels")
    if raw_labels is None:
        return None, "Token Position"

    stripped_labels = [label.strip() for label in raw_labels]
    leading_bos_tokens = {"<|endoftext|>", "<bos>", "<s>"}
    template_offset = 1 if stripped_labels and stripped_labels[0].lower() in leading_bos_tokens else 0
    template_slice = stripped_labels[template_offset:]

    if (
        len(template_slice) == 6
        and template_slice[0].lower() == "the"
        and template_slice[2].lower() == "was"
        and template_slice[4].lower() == "by"
        and template_slice[5].lower() == "the"
    ):
        template_labels = template_slice.copy()
        template_labels[1] = "[patient]"
        template_labels[3] = "[verb]"
        if template_offset == 1:
            return [stripped_labels[0], *template_labels], "Template Position"
        return template_labels, "Template Position"

    return stripped_labels, "Token Position"

def get_saved_token_index(data, token_label, occurrence=0):
    plot_labels, _ = get_saved_heatmap_x_labels(data)
    matches = [
        idx for idx, label in enumerate(plot_labels)
        if label.strip().lower() == token_label.strip().lower()
    ]
    if not matches:
        raise ValueError(f"Could not find token '{token_label}' in labels: {plot_labels}")
    try:
        return matches[occurrence]
    except IndexError as exc:
        raise ValueError(
            f"Token '{token_label}' does not have occurrence {occurrence} in labels: {plot_labels}"
        ) from exc

def draw_heatmap(file_path):
    data = torch.load(file_path)
    
    normalized_map = get_saved_normalized_heatmap(data)
    x_labels, x_axis_title = get_saved_heatmap_x_labels(data)
    
    fig = px.imshow(
        normalized_map.numpy(), 
        labels={"x": x_axis_title, "y": "Layer"},
        x=x_labels,
        title=f"Results: {data['experiment']}",
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0.0
    )
    fig.update_yaxes(autorange="reversed") 
    image_dir = Path(file_path).parent / "img"
    image_dir.mkdir(parents=True, exist_ok=True)
    fig.write_image(str(image_dir / f"{data['experiment']}_head_heatmap.png"))
    fig.show()

def draw_head_heatmap(file_path, token_idx=-1):
    data = torch.load(file_path)
    normalized_heatmap = get_saved_normalized_heatmap(data)
    
    # We slice it to look at ONLY the final token position
    # Shape becomes [layers, heads]
    normalized_slice = normalized_heatmap[:, :, token_idx]
    
    fig = px.imshow(
        normalized_slice.numpy(),
        labels={"x": "Head Index", "y": "Layer"},
        title=f"Head Patching at Token Index {token_idx} ({data['experiment']})",
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0.0
    )
    fig.update_yaxes(autorange="reversed")
    image_dir = Path(file_path).parent / "img"
    image_dir.mkdir(parents=True, exist_ok=True)
    fig.write_image(str(image_dir / f"{data['experiment']}_head_heatmap.png"))
    fig.show()

In [16]:
metric_factory = make_avg_logit_difference_recovery_metric(animate_ids_tensor, inanimate_ids_tensor)

resid_stream_path_file = run_and_save_experiment(
    experiment_name="Residual_Stream_Patching", 
    df=dataset_filtered, 
    tokenizer=tokenizer,
    patching_func=patching.get_act_patch_resid_pre, 
    metric_factory=metric_factory, 
    filter_str="hook_resid_pre",
    batch_size=BATCH_SIZE,
    safety_checks=True
)

--- Starting Experiment: Residual_Stream_Patching ---
Starting batched patching over 500 samples. Max sequence length: 7


Patching generic_activation_patch:   0%|          | 0/500 [00:00<?, ?seq/s]

[DEBUG] Group Length 7 | Example: 'The valuation was managed by the'
[DEBUG]        ---> logits[:, -1, :] predicts the NEXT token after the final input token ' the'



  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/84 [00:00<?, ?it/s]

...

  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/84 [00:00<?, ?it/s]

Saved successfully to C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\results\2026-05-15\Residual_Stream_Patching_2026-05-15.pt



In [17]:
resid_stream_path_file = os.path.join(working_dir, "results", "2026-05-15", "Residual_Stream_Patching_2026-05-15.pt")
draw_heatmap(resid_stream_path_file)

In [18]:
data = torch.load(resid_stream_path_file)
plot_labels, _ = get_saved_heatmap_x_labels(data)
resid_heatmap = get_saved_normalized_heatmap(data)

verb_idx = next(
    (idx for idx, label in enumerate(plot_labels) if label.strip().lower() == "[verb]"),
    None,
)

if verb_idx is None:
    raise ValueError(f"Could not find '[verb]' in labels: {plot_labels}")

verb_onward_token_indices = list(range(verb_idx, resid_heatmap.shape[1]))

rows = [{
    "token_position": token_idx,
    "token": plot_labels[token_idx],
    "token_label": f"{token_idx}: {plot_labels[token_idx]}",
    "token_position_from_end": token_idx - resid_heatmap.shape[1],
    "layer": layer_idx,
    "score": resid_heatmap[layer_idx, token_idx].item()
} for token_idx in verb_onward_token_indices for layer_idx in range(resid_heatmap.shape[0])]

verb_onward_df = pd.DataFrame(rows)

layer_token_scores = (
    verb_onward_df
    .pivot(index="layer", columns="token_label", values="score")
    .sort_index()
)

verb_onward_df["abs_score"] = verb_onward_df["score"].abs()
verb_onward_df["token_threshold"] = (
    verb_onward_df
    .groupby("token_position")["abs_score"]
    .transform(lambda scores: scores.quantile(0.10))
)

scores_by_token = [{
    "token_position": token_idx,
    "token": plot_labels[token_idx],
    "token_position_from_end": token_idx - resid_heatmap.shape[1],
    "threshold": (
        verb_onward_df.loc[
            verb_onward_df["token_position"] == token_idx,
            "token_threshold",
        ].iloc[0]
    ),
    "layer_scores": (
        verb_onward_df[verb_onward_df["token_position"] == token_idx]
        .sort_values("score", ascending=False)[["layer", "score"]]
        .to_dict("records")
    )
} for token_idx in verb_onward_token_indices]

high_score_df = (
    verb_onward_df[verb_onward_df["score"] > verb_onward_df["token_threshold"]]
    .sort_values(["token_position", "score"], ascending=[True, False])
    .reset_index(drop=True)
)

display(layer_token_scores)
display(high_score_df)
scores_by_token

token_label,4: [verb],5: by,6: the
layer,,,
0,1.000000,0.000000,0.000000
1,0.992928,0.021230,0.005577
2,0.984119,0.029734,0.009346
3,0.921547,0.162666,0.027275
4,0.696654,0.400041,0.049260
5,0.423569,0.494577,0.213120
6,0.203589,0.510027,0.410257
7,0.105653,0.467701,0.563311
8,0.004168,0.348096,0.695303


,token_position,token,token_label,token_position_from_end,layer,score,abs_score,token_threshold
0,4,[verb],4: [verb],-3,0,1.000000,1.000000,0.004283
1,4,[verb],4: [verb],-3,1,0.992928,0.992928,0.004283
2,4,[verb],4: [verb],-3,2,0.984119,0.984119,0.004283
3,4,[verb],4: [verb],-3,3,0.921547,0.921547,0.004283
4,4,[verb],4: [verb],-3,4,0.696654,0.696654,0.004283
5,4,[verb],4: [verb],-3,5,0.423569,0.423569,0.004283
6,4,[verb],4: [verb],-3,6,0.203589,0.203589,0.004283
7,4,[verb],4: [verb],-3,7,0.105653,0.105653,0.004283
8,5,by,5: by,-2,6,0.510027,0.510027,0.005437
9,5,by,5: by,-2,5,0.494577,0.494577,0.005437


[{'token_position': 4,
  'token': '[verb]',
  'token_position_from_end': -3,
  'threshold': 0.004282789723947644,
  'layer_scores': [{'layer': 0, 'score': 1.0},
   {'layer': 1, 'score': 0.9929276704788208},
   {'layer': 2, 'score': 0.9841192364692688},
   {'layer': 3, 'score': 0.9215472340583801},
   {'layer': 4, 'score': 0.6966543197631836},
   {'layer': 5, 'score': 0.4235692024230957},
   {'layer': 6, 'score': 0.20358876883983612},
   {'layer': 7, 'score': 0.10565284639596939},
   {'layer': 8, 'score': 0.00416812626644969},
   {'layer': 9, 'score': -0.0018592126434668899},
   {'layer': 11, 'score': -0.0053147608414292336},
   {'layer': 10, 'score': -0.005480571184307337}]},
 {'token_position': 5,
  'token': 'by',
  'token_position_from_end': -2,
  'threshold': 0.005437423940747977,
  'layer_scores': [{'layer': 6, 'score': 0.5100274085998535},
   {'layer': 5, 'score': 0.4945768117904663},
   {'layer': 7, 'score': 0.46770063042640686},
   {'layer': 4, 'score': 0.400040864944458},
   {'

<h3>Activation patching on Attention and MLP modules </h3>

In [19]:
def make_targeted_patching_func(layers_to_patch, pos_from_end, component="mlp_out"):
    """
    Creates a patching function that only runs hooks for specific layers at a specific
    token position (relative to the end of the sequence).
    """
    def custom_patching_func(model, corrupted_tokens, clean_cache, patching_metric):
        seq_len = corrupted_tokens.shape[1]
        n_layers = model.cfg.n_layers
        
        # Initialize results grid with zeros
        results = torch.zeros((n_layers, seq_len), device=model.cfg.device)
        
        pos = seq_len + pos_from_end
        if pos < 0 or pos >= seq_len:
            return results

        for layer in layers_to_patch:
            hook_name = f"blocks.{layer}.hook_{component}"
            
            def single_pos_hook_fn(corrupt_act, hook, target_pos=pos):
                clean_act = clean_cache[hook.name]
                # Replace just the activation at the targeted position
                corrupt_act[:, target_pos, :] = clean_act[:, target_pos, :]
                return corrupt_act
                
            # Run model with hook
            patched_logits = model.run_with_hooks(
                corrupted_tokens,
                fwd_hooks=[(hook_name, single_pos_hook_fn)],
                return_type="logits"
            )
            
            results[layer, pos] = patching_metric(patched_logits).item()
            
        return results
        
    return custom_patching_func

<h2>MLP and Attention patching for [verb] tokens</h2>

In [20]:
metric_factory = make_avg_logit_difference_recovery_metric(animate_ids_tensor, inanimate_ids_tensor)

# Filter for only the [verb] token positions
verb_df = high_score_df[high_score_df["token"].str.strip().str.lower() == "[verb]"]

# Extract layers to patch and pos_from_end (assuming it's consistent for the group)
verb_layers_to_patch = verb_df["layer"].astype(int).tolist()
pos_from_end = int(verb_df["token_position_from_end"].iloc[0]) if not verb_df.empty else 0

In [21]:
# --- 2. MLP Component Patching on [verb] ---
targeted_mlp_patching_verb = make_targeted_patching_func(verb_layers_to_patch, pos_from_end, component="mlp_out")

# This runs the targeted loop, saves the results, and returns the path
mlp_results_file_verb = run_and_save_experiment(
    experiment_name="MLP_Outputs_Targeted_Verb",
    df=dataset_filtered,
    tokenizer=tokenizer,
    patching_func=targeted_mlp_patching_verb,
    metric_factory=metric_factory,
    filter_str="hook_mlp_out",
    batch_size=BATCH_SIZE,
    safety_checks=True
)

--- Starting Experiment: MLP_Outputs_Targeted_Verb ---
Starting batched patching over 500 samples. Max sequence length: 7


Patching custom_patching_func:   0%|          | 0/500 [00:00<?, ?seq/s]

[DEBUG] Group Length 7 | Example: 'The valuation was managed by the'
[DEBUG]        ---> logits[:, -1, :] predicts the NEXT token after the final input token ' the'

Saved successfully to C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\results\2026-05-15\MLP_Outputs_Targeted_Verb_2026-05-15.pt



In [22]:
# draw_heatmap(mlp_results_file_verb)

verb_mlp_results = torch.load(mlp_results_file_verb)
verb_token_idx = get_saved_token_index(verb_mlp_results, "[verb]")

print("MLP Patching Results for [verb] token positions:")
verb_layers_to_patch.sort()
for i in verb_layers_to_patch:
    print(f"L{i}: {verb_mlp_results['normalized_heatmap'][i, verb_token_idx]}")

MLP Patching Results for [verb] token positions:
L0: 0.8963066339492798
L1: 0.06459789723157883
L2: 0.061809949576854706
L3: 0.06571412831544876
L4: 0.04327049478888512
L5: -0.01872124895453453
L6: 0.018987972289323807
L7: -0.0029306886717677116


In [23]:
# --- 3. Attention Component Patching on [verb] ---

targeted_attn_patching_verb = make_targeted_patching_func(verb_layers_to_patch, pos_from_end, component="attn_out")

attn_results_file_verb = run_and_save_experiment(
    experiment_name="Attention_Outputs_Targeted_Verb",
    df=dataset_filtered,
    tokenizer=tokenizer,
    patching_func=targeted_attn_patching_verb,
    metric_factory=metric_factory,
    filter_str="hook_attn_out",
    batch_size=BATCH_SIZE,
    safety_checks=True
)

--- Starting Experiment: Attention_Outputs_Targeted_Verb ---
Starting batched patching over 500 samples. Max sequence length: 7


Patching custom_patching_func:   0%|          | 0/500 [00:00<?, ?seq/s]

[DEBUG] Group Length 7 | Example: 'The valuation was managed by the'
[DEBUG]        ---> logits[:, -1, :] predicts the NEXT token after the final input token ' the'

Saved successfully to C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\results\2026-05-15\Attention_Outputs_Targeted_Verb_2026-05-15.pt



In [24]:
# draw_heatmap(attn_results_file_verb)

attn_verb_results = torch.load(attn_results_file_verb)
verb_token_idx = get_saved_token_index(attn_verb_results, "[verb]")

print("Attention Patching Results for [verb] token positions:")
verb_layers_to_patch.sort()
for i in verb_layers_to_patch:
    print(f"L{i}: {attn_verb_results['normalized_heatmap'][i, verb_token_idx]}")

Attention Patching Results for [verb] token positions:
L0: 0.08425505459308624
L1: 0.008714988827705383
L2: 0.00059128477005288
L3: 0.00256594130769372
L4: -0.0064203767105937
L5: 0.0011004414409399033
L6: -0.0014482748229056597
L7: 0.00014045606076251715


<h2>MLP and Attention patching for [by] tokens</h2>

In [25]:
metric_factory = make_avg_logit_difference_recovery_metric(animate_ids_tensor, inanimate_ids_tensor)

# Filter for only the [verb] token positions
by_df = high_score_df[high_score_df["token"].str.strip().str.lower() == "by"]

# Extract layers to patch and pos_from_end (assuming it's consistent for the group)
by_layers_to_patch = by_df["layer"].astype(int).tolist()
pos_from_end = int(by_df["token_position_from_end"].iloc[0]) if not by_df.empty else 0

In [27]:
# --- 2. MLP Component Patching on [verb] ---
targeted_mlp_patching_by = make_targeted_patching_func(by_layers_to_patch, pos_from_end, component="mlp_out")

# This runs the targeted loop, saves the results, and returns the path
mlp_results_file_by = run_and_save_experiment(
    experiment_name="MLP_Outputs_Targeted_by",
    df=dataset_filtered,
    tokenizer=tokenizer,
    patching_func=targeted_mlp_patching_by,
    metric_factory=metric_factory,
    filter_str="hook_mlp_out",
    batch_size=BATCH_SIZE,
    safety_checks=True
)

--- Starting Experiment: MLP_Outputs_Targeted_by ---
Starting batched patching over 500 samples. Max sequence length: 7


Patching custom_patching_func:   0%|          | 0/500 [00:00<?, ?seq/s]

[DEBUG] Group Length 7 | Example: 'The valuation was managed by the'
[DEBUG]        ---> logits[:, -1, :] predicts the NEXT token after the final input token ' the'

Saved successfully to C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\results\2026-05-15\MLP_Outputs_Targeted_by_2026-05-15.pt



In [28]:
# draw_heatmap(mlp_results_file_by)

by_mlp_results = torch.load(mlp_results_file_by)
by_token_idx = get_saved_token_index(by_mlp_results, "by")

print("MLP Patching Results for [by] token positions:")
by_layers_to_patch.sort()
for i in by_layers_to_patch:
    print(f"L{i}: {by_mlp_results['normalized_heatmap'][i, by_token_idx]}")

MLP Patching Results for [by] token positions:
L1: -0.010378524661064148
L2: -0.013889125548303127
L3: 0.14687664806842804
L4: 0.028048397973179817
L5: 0.08563300222158432
L6: 0.04730090871453285
L7: 0.03913699835538864
L8: 0.06082604080438614
L9: 0.008054005913436413
L10: 0.0025386742781847715


In [29]:
metric_factory = make_avg_logit_difference_recovery_metric(animate_ids_tensor, inanimate_ids_tensor)

targeted_attn_patching_by = make_targeted_patching_func(by_layers_to_patch, pos_from_end, component="attn_out")

attn_results_file_by = run_and_save_experiment(
    experiment_name="Attention_Outputs_Targeted_by",
    df=dataset_filtered,
    tokenizer=tokenizer,
    patching_func=targeted_attn_patching_by,
    metric_factory=metric_factory,
    filter_str="hook_attn_out",
    batch_size=BATCH_SIZE,
    safety_checks=True
)

--- Starting Experiment: Attention_Outputs_Targeted_by ---
Starting batched patching over 500 samples. Max sequence length: 7


Patching custom_patching_func:   0%|          | 0/500 [00:00<?, ?seq/s]

[DEBUG] Group Length 7 | Example: 'The valuation was managed by the'
[DEBUG]        ---> logits[:, -1, :] predicts the NEXT token after the final input token ' the'

Saved successfully to C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\results\2026-05-15\Attention_Outputs_Targeted_by_2026-05-15.pt



In [30]:
# draw_heatmap(attn_results_file_by)

by_attn_results = torch.load(attn_results_file_by)
by_token_idx = get_saved_token_index(by_attn_results, "by")

print("Attention Patching Results for [by] token positions:")
by_layers_to_patch.sort()
for i in by_layers_to_patch:
    print(f"L{i}: {by_attn_results['normalized_heatmap'][i, by_token_idx]}")

Attention Patching Results for [by] token positions:
L1: 0.004983203951269388
L2: 0.10624867677688599
L3: 0.10323293507099152
L4: 0.037494804710149765
L5: 0.06290872395038605
L6: 0.02317460998892784
L7: 0.061297234147787094
L8: 0.013964169658720493
L9: 0.0012291425373405218
L10: -0.001242751837708056


<h2>MLP and Attention patching for [the] tokens</h2>

In [31]:
metric_factory = make_avg_logit_difference_recovery_metric(animate_ids_tensor, inanimate_ids_tensor)

# Filter for only the [verb] token positions
the_df = high_score_df[high_score_df["token"].str.strip().str.lower() == "the"]

# Extract layers to patch and pos_from_end (assuming it's consistent for the group)
the_layers_to_patch = the_df["layer"].astype(int).tolist()
pos_from_end = int(the_df["token_position_from_end"].iloc[0]) if not the_df.empty else 0

In [32]:

# --- 2. MLP Component Patching on [verb] ---
targeted_mlp_patching_the = make_targeted_patching_func(the_layers_to_patch, pos_from_end, component="mlp_out")

# This runs the targeted loop, saves the results, and returns the path
the_mlp_results_file_the = run_and_save_experiment(
    experiment_name="MLP_Outputs_Targeted_the",
    df=dataset_filtered,
    tokenizer=tokenizer,
    patching_func=targeted_mlp_patching_the,
    metric_factory=metric_factory,
    filter_str="hook_mlp_out",
    batch_size=BATCH_SIZE,
    safety_checks=True
)

--- Starting Experiment: MLP_Outputs_Targeted_the ---
Starting batched patching over 500 samples. Max sequence length: 7


Patching custom_patching_func:   0%|          | 0/500 [00:00<?, ?seq/s]

[DEBUG] Group Length 7 | Example: 'The valuation was managed by the'
[DEBUG]        ---> logits[:, -1, :] predicts the NEXT token after the final input token ' the'

Saved successfully to C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\results\2026-05-15\MLP_Outputs_Targeted_the_2026-05-15.pt



In [33]:
# draw_heatmap(mlp_results_file_the)

the_mlp_results = torch.load(mlp_results_file_the)
the_token_idx = get_saved_token_index(the_mlp_results, "the", occurrence=-1)

print("MLP Patching Results for [the] token positions:")
the_layers_to_patch.sort()
for i in the_layers_to_patch:
    print(f"L{i}: {the_mlp_results['normalized_heatmap'][i, the_token_idx]}")

MLP Patching Results for [the] token positions:
L2: 0.0
L3: 0.0
L4: 0.0
L5: 0.0
L6: 0.027762023732066154
L7: 0.02413155511021614
L8: 0.1774575263261795
L9: 0.14945577085018158
L10: 0.14893782138824463
L11: 0.060018621385097504


In [34]:
metric_factory = make_avg_logit_difference_recovery_metric(animate_ids_tensor, inanimate_ids_tensor)

targeted_attn_patching_the = make_targeted_patching_func(the_layers_to_patch, pos_from_end, component="attn_out")

the_attn_results_file_the = run_and_save_experiment(
    experiment_name="Attention_Outputs_Targeted_the",
    df=dataset_filtered,
    tokenizer=tokenizer,
    patching_func=targeted_attn_patching_the,
    metric_factory=metric_factory,
    filter_str="hook_attn_out",
    batch_size=BATCH_SIZE,
    safety_checks=True
)

--- Starting Experiment: Attention_Outputs_Targeted_the ---
Starting batched patching over 500 samples. Max sequence length: 7


Patching custom_patching_func:   0%|          | 0/500 [00:00<?, ?seq/s]

[DEBUG] Group Length 7 | Example: 'The valuation was managed by the'
[DEBUG]        ---> logits[:, -1, :] predicts the NEXT token after the final input token ' the'

Saved successfully to C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\results\2026-05-15\Attention_Outputs_Targeted_the_2026-05-15.pt



In [35]:
# draw_heatmap(the_attn_results_file_the)

the_attn_results = torch.load(attn_results_file_the)
the_token_idx = get_saved_token_index(the_attn_results, "the", occurrence=-1)

print("Attention Patching Results for [the] token positions:")
the_layers_to_patch.sort()
for i in the_layers_to_patch:
    print(f"L{i}: {the_attn_results['normalized_heatmap'][i, the_token_idx]}")

Attention Patching Results for [the] token positions:
L2: 0.0
L3: 0.0
L4: 0.0
L5: 0.0
L6: 0.07086249440908432
L7: 0.0939314216375351
L8: 0.16083617508411407
L9: 0.1398293524980545
L10: 0.0722370520234108
L11: 0.0058875922113657


In [36]:
RESULTS_DIR = Path(working_dir) / "results"
IMG_DIR = RESULTS_DIR / datetime.datetime.now().strftime("%Y-%m-%d") / "img"
IMG_DIR.mkdir(parents=True, exist_ok=True)

aggregate_specs = {
    "MLP": [
        {"token_label": "[verb]", "var_name": "verb_mlp_results_file_verb", "pattern": "MLP_Outputs_Targeted_Verb_*.pt"},
        {"token_label": "by", "var_name": "by_mlp_results_file_verb", "pattern": "MLP_Outputs_Targeted_by_*.pt"},
        {"token_label": "the", "occurrence": -1, "var_name": "the_mlp_results_file_verb", "pattern": "MLP_Outputs_Targeted_the_*.pt"},
    ],
    "Attention Output": [
        {"token_label": "[verb]", "var_name": "verb_attn_results_file_verb", "pattern": "Attention_Outputs_Targeted_Verb_*.pt"},
        {"token_label": "by", "var_name": "by_attn_results_file_verb", "pattern": "Attention_Outputs_Targeted_by_*.pt"},
        {"token_label": "the", "occurrence": -1, "var_name": "the_attn_results_file_verb", "pattern": "Attention_Outputs_Targeted_the_*.pt"},
    ],
}

def resolve_result_path(var_name, pattern):
    candidate = globals().get(var_name)
    if candidate is not None:
        candidate = Path(candidate)
        if candidate.exists():
            return candidate

    matches = sorted((path for path in RESULTS_DIR.rglob(pattern) if path.is_file()), key=lambda path: path.stat().st_mtime)
    if not matches:
        raise FileNotFoundError(f"Could not find a results file matching '{pattern}'")

    return matches[-1]

def build_aggregate_heatmap(specs):
    aggregate_columns = []
    source_rows = []

    for spec in specs:
        file_path = resolve_result_path(spec["var_name"], spec["pattern"])
        data = torch.load(file_path)
        heatmap = get_saved_normalized_heatmap(data)
        plot_labels, _ = get_saved_heatmap_x_labels(data)

        token_idx = get_saved_token_index(
            data,
            spec["token_label"],
            occurrence=spec.get("occurrence", 0),
        )

        aggregate_columns.append(heatmap[:, token_idx])
        source_rows.append(
            {
                "token": plot_labels[token_idx],
                "token_position": token_idx,
                "source_file": file_path.name,
            }
        )

    aggregate_tensor = torch.stack(aggregate_columns, dim=1)
    aggregate_df = pd.DataFrame(
        aggregate_tensor.numpy(),
        index=range(aggregate_tensor.shape[0]),
        columns=[row["token"] for row in source_rows],
    )
    aggregate_df.index.name = "layer"

    return aggregate_df, pd.DataFrame(source_rows)

def draw_aggregate_component_heatmap(component_name, specs):
    aggregate_df, source_df = build_aggregate_heatmap(specs)

    fig = px.imshow(
        aggregate_df.to_numpy(),
        labels={"x": "Token", "y": "Layer"},
        x=aggregate_df.columns.tolist(),
        y=aggregate_df.index.tolist(),
        title=f"Aggregate {component_name} targeted heatmap",
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0.0
    )
    fig.update_yaxes(autorange="reversed")
    fig.write_image(str(IMG_DIR / f"{component_name.replace(' ', '_')}_Targeted_Aggregate_heatmap.png"))
    fig.show()

    return aggregate_df, source_df

mlp_aggregate_df, mlp_aggregate_sources = draw_aggregate_component_heatmap("MLP", aggregate_specs["MLP"])
attn_aggregate_df, attn_aggregate_sources = draw_aggregate_component_heatmap("Attention Output", aggregate_specs["Attention Output"])


<h2>Head patching for all the three token positions</h2>

In [38]:
def make_targeted_head_patching_func(layer_map_by_pos_from_end):
    """
    layer_map_by_pos_from_end:
        dict mapping token position from end -> list of layers to patch

    Returns a patching function with output shape [layers, heads, pos],
    compatible with run_and_save_experiment + draw_head_heatmap.
    """
    def custom_head_patching_func(model, corrupted_tokens, clean_cache, patching_metric):
        seq_len = corrupted_tokens.shape[1]
        n_layers = model.cfg.n_layers
        n_heads = model.cfg.n_heads

        results = torch.zeros((n_layers, n_heads, seq_len), device=model.cfg.device)

        for pos_from_end, layers_to_patch in layer_map_by_pos_from_end.items():
            pos = seq_len + pos_from_end
            if pos < 0 or pos >= seq_len:
                continue

            for layer in layers_to_patch:
                hook_name = f"blocks.{layer}.attn.hook_z"

                for head in range(n_heads):
                    def single_head_hook_fn(corrupt_act, hook, target_pos=pos, target_head=head):
                        clean_act = clean_cache[hook.name]
                        corrupt_act[:, target_pos, target_head, :] = clean_act[:, target_pos, target_head, :]
                        return corrupt_act

                    patched_logits = model.run_with_hooks(
                        corrupted_tokens,
                        fwd_hooks=[(hook_name, single_head_hook_fn)],
                        return_type="logits",
                    )

                    results[layer, head, pos] = patching_metric(patched_logits).item()

        return results

    return custom_head_patching_func

In [ ]:
DISCOVERED_COMPONENT_SPECS = (
    {
        "name": "verb",
        "token_label": "[verb]",
        "occurrence": 0,
        "mlp_filename": "MLP_Outputs_Targeted_Verb_{day}.pt",
        "attn_filename": "Attention_Outputs_Targeted_Verb_{day}.pt",
    },
    {
        "name": "by",
        "token_label": "by",
        "occurrence": 0,
        "mlp_filename": "MLP_Outputs_Targeted_by_{day}.pt",
        "attn_filename": "Attention_Outputs_Targeted_by_{day}.pt",
    },
    {
        "name": "the",
        "token_label": "the",
        "occurrence": -1,
        "mlp_filename": "MLP_Outputs_Targeted_the_{day}.pt",
        "attn_filename": "Attention_Outputs_Targeted_the_{day}.pt",
    },
)


def resolve_day_result_path(results_dir, day, filename_template):
    file_path = results_dir / day / filename_template.format(day=day)
    if not file_path.is_file():
        raise FileNotFoundError(f"Could not find checkpoint: {file_path}")
    return file_path


def heatmap_to_score_rows(data):
    heatmap = get_saved_normalized_heatmap(data)
    if heatmap.ndim != 2:
        raise ValueError(f"Expected a 2D heatmap, got shape {tuple(heatmap.shape)}")

    labels, _ = get_saved_heatmap_x_labels(data)
    if labels is None:
        labels = [str(index) for index in range(heatmap.shape[1])]

    rows = []
    for token_position in range(heatmap.shape[1]):
        token = labels[token_position]
        pos_from_end = token_position - heatmap.shape[1]
        for layer in range(heatmap.shape[0]):
            rows.append({
                "layer": layer,
                "token_position": token_position,
                "token_position_from_end": pos_from_end,
                "token": token,
                "score": float(heatmap[layer, token_position].item()),
            })
    return pd.DataFrame(rows)


def _token_position_from_end(data, token_label, occurrence):
    token_idx = get_saved_token_index(data, token_label, occurrence=occurrence)
    return int(token_idx - get_saved_normalized_heatmap(data).shape[1])


def _positive_token_layer_rows(data, token_label, occurrence, min_score=0.0):
    rows = heatmap_to_score_rows(data)
    token_pos_from_end = _token_position_from_end(data, token_label, occurrence)
    selected = rows[
        (rows["token_position_from_end"] == token_pos_from_end)
        & (rows["score"] > min_score)
    ].copy()
    return selected.sort_values("score", ascending=False).reset_index(drop=True)


def build_head_layer_targets(results_dir, day, min_score=0.0):
    head_layer_targets = {}
    summary_rows = []

    for spec in DISCOVERED_COMPONENT_SPECS:
        file_path = resolve_day_result_path(results_dir, day, spec["attn_filename"])
        data = torch.load(file_path)
        pos_from_end = _token_position_from_end(data, spec["token_label"], spec["occurrence"])
        positive_rows = _positive_token_layer_rows(
            data=data,
            token_label=spec["token_label"],
            occurrence=spec["occurrence"],
            min_score=min_score,
        )
        selected_layers = sorted({int(layer) for layer in positive_rows["layer"].tolist()})
        head_layer_targets[pos_from_end] = selected_layers

        for row in positive_rows.itertuples(index=False):
            summary_rows.append({
                "source": spec["name"],
                "component_type": "attn_layer",
                "token": spec["token_label"],
                "token_position_from_end": int(row.token_position_from_end),
                "layer": int(row.layer),
                "score": float(row.score),
                "checkpoint": str(file_path),
            })

    summary_df = pd.DataFrame(summary_rows)
    if not summary_df.empty:
        summary_df = summary_df.sort_values(
            ["token_position_from_end", "score"],
            ascending=[True, False],
        ).reset_index(drop=True)
    return head_layer_targets, summary_df


metric_factory = make_avg_logit_difference_recovery_metric(
    animate_ids_tensor,
    inanimate_ids_tensor,
)

HEAD_PATCH_DAY = datetime.datetime.now().strftime("%Y-%m-%d")
HEAD_RESULTS_DIR = Path(working_dir) / "results"
HEAD_RESULTS_DAY_DIR = HEAD_RESULTS_DIR / HEAD_PATCH_DAY

head_layer_targets, head_layer_summary = build_head_layer_targets(
    results_dir=HEAD_RESULTS_DIR,
    day=HEAD_PATCH_DAY,
    min_score=0.0,
)
display(head_layer_summary)

if not any(head_layer_targets.values()):
    raise ValueError(f"No positive attention-output layers were found in {HEAD_RESULTS_DAY_DIR}.")

head_checkpoint_path = HEAD_RESULTS_DAY_DIR / f"Attention_Heads_Targeted_all_{HEAD_PATCH_DAY}.pt"
if head_checkpoint_path.is_file():
    attention_head_targeted_results_file = str(head_checkpoint_path)
else:
    targeted_head_patching = make_targeted_head_patching_func(head_layer_targets)
    attention_head_targeted_results_file = run_and_save_experiment(
        experiment_name="Attention_Heads_Targeted_all",
        df=dataset_filtered,
        tokenizer=tokenizer,
        patching_func=targeted_head_patching,
        metric_factory=metric_factory,
        filter_str="attn.hook_z",
        batch_size=BATCH_SIZE,
        safety_checks=True,
    )

draw_head_heatmap(attention_head_targeted_results_file, token_idx=-3)
draw_head_heatmap(attention_head_targeted_results_file, token_idx=-2)
draw_head_heatmap(attention_head_targeted_results_file, token_idx=-1)


--- Starting Experiment: Attention_Heads_Targeted_all ---
Starting batched patching over 500 samples. Max sequence length: 7


Patching custom_head_patching_func:   0%|          | 0/500 [00:00<?, ?seq/s]

[DEBUG] Group Length 7 | Example: 'The valuation was managed by the'
[DEBUG]        ---> logits[:, -1, :] predicts the NEXT token after the final input token ' the'

Saved successfully to c:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\scripts\../results/Attention_Heads_Targeted_all_2026-05-08.pt



In [81]:
attention_head_targeted = torch.load(attention_head_targeted_results_file)
head_token_names = {-3: "[verb]", -2: "by", -1: "the"}

print("Attention Head Patching Results for targeted token positions:")
for pos_from_end, layers in sorted(head_layer_targets.items()):
    if not layers:
        continue
    token_name = head_token_names.get(pos_from_end, str(pos_from_end))
    print(
        f"Layers {layers} for {token_name}:\n",
        attention_head_targeted["normalized_heatmap"][layers, :, pos_from_end],
    )

Attention Head Patching Results for targeted token positions:
Layers 0-4 for [verb]:
 tensor([[ 6.5410e-03,  3.7545e-03,  1.8258e-03,  1.9662e-02,  2.8751e-02,
          2.8537e-03, -6.4297e-04, -1.3171e-04,  1.2853e-03, -9.7061e-04,
          2.6838e-05, -1.3109e-03],
        [-7.5376e-04, -1.0412e-03, -1.2969e-04,  2.8622e-04, -2.7270e-04,
          3.5630e-04,  8.4635e-04, -9.1080e-05,  3.9006e-04,  2.7227e-04,
          9.7519e-05,  1.0072e-02],
        [-2.3270e-03,  4.1882e-03, -3.3153e-04, -1.3276e-03, -1.5936e-03,
          1.6278e-03, -1.1406e-03, -2.0323e-03, -6.6644e-04,  9.5365e-04,
          5.5375e-03, -2.0811e-03],
        [ 1.5952e-03,  6.0156e-04, -7.7713e-05, -1.0115e-03,  1.8777e-04,
          3.3501e-03, -1.0980e-03,  1.9030e-03,  3.9847e-04, -5.3267e-04,
         -8.0896e-03,  4.5420e-03]])
Layers 4-7 for [by]:
 tensor([[ 1.2971e-02,  8.2435e-04, -2.2363e-03,  2.8823e-03, -4.2443e-03,
          1.1428e-03, -6.3904e-03, -1.1015e-03,  2.6246e-04,  2.1912e-02,
       